# 01_data_collection — データ収集

## モード
| `DATA_SOURCE_MODE` | 動作 |
|---|---|
| `existing` | 既存 keiba_ultimate.db の統計を表示 |
| `simulation` | `_run_scrape_job` を直接呼び出して sim DB にデータ収集 |

## フロントエンド対応表
| フロント | Notebook |
|---|---|
| `POST /api/scrape/start` + polling | `asyncio.SelectorEventLoop` + `threading.Thread` |
| `GET /api/data-stats` | `db_stats()` 直接呼び出し |
| `GET /api/races/recent` | SQLite 直接クエリ |
| useBatchScrape Hook | Cell DC-3 のポーリングループ |

In [1]:
# ── ブートストラップ ──────────────────────────────────────────
import sys, json
from pathlib import Path

_NB_DIR = Path().resolve()
if _NB_DIR.name != 'notebooks':
    _NB_DIR = _NB_DIR.parent
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from utils.nb_config import *

# config.json があれば読み込む
_cfg_file = NB_DIR / 'config.json'
if _cfg_file.exists():
    _cfg = json.loads(_cfg_file.read_text(encoding='utf-8'))
    DATA_SOURCE_MODE = _cfg.get('DATA_SOURCE_MODE', DATA_SOURCE_MODE)
    SIM_SCRAPE_START = _cfg.get('SIM_SCRAPE_START', '2026-01')
    SIM_SCRAPE_END   = _cfg.get('SIM_SCRAPE_END',   '2026-02')

print(f"DATA_SOURCE_MODE = {DATA_SOURCE_MODE}")

DATA_SOURCE_MODE = existing


## 現状確認（existing / simulation 共通）

In [2]:
# ── DB 統計表示 ───────────────────────────────────────────────
import sqlite3, pandas as pd

_db = get_db_path(DATA_SOURCE_MODE)
print(f"対象 DB: {_db}")
print(f"DB 存在: {_db.exists()}")

s = db_stats(_db)
print("\n=== DB 統計 ===")
for k, v in s.items():
    print(f"  {k:15s}: {v}")

# 最近取得したレース（上位20件）
if _db.exists():
    conn = sqlite3.connect(str(_db))
    try:
        _rows = conn.execute(
            "SELECT race_id, data FROM races_ultimate ORDER BY race_id DESC LIMIT 20"
        ).fetchall()
        import json as _json
        _recent = []
        for rid, dstr in _rows:
            try:
                d = _json.loads(dstr)
                _recent.append({'race_id': rid, 'date': d.get('date',''), 'venue': d.get('venue',''),
                                 'race_name': (d.get('race_name') or d.get('name',''))[:20], 'n': d.get('num_horses',0)})
            except Exception: pass
        _df_r = pd.DataFrame(_recent)
        print("\n=== 最近取得レース（上位20件）===")
        display(_df_r) if '_df_r' in dir() and len(_df_r) > 0 else print(_df_r.to_string())
    finally:
        conn.close()

対象 DB: C:\Users\yuki2\Documents\ws\keiba-ai-pro\keiba\data\keiba_ultimate.db
DB 存在: True

=== DB 統計 ===
  races          : 52450
  results        : 575346
  latest         : 202665060812
  scraped_dates  : 797

=== 最近取得レース（上位20件）===


,race_id,date,venue,race_name,n
0,202665060812,20260608,帯広(ばんえい),B4ー1,10
1,202665060811,20260608,帯広(ばんえい),シルバーカップ,10
2,202665060810,20260608,帯広(ばんえい),夏さん・凜さん帯広上,8
3,202665060809,20260608,帯広(ばんえい),耕司、里香子結婚7周,10
4,202665060808,20260608,帯広(ばんえい),祝!たかし還暦おめで,9
5,202665060807,20260608,帯広(ばんえい),ねやちゃん記念ねやC,10
6,202665060806,20260608,帯広(ばんえい),めっちゃん生誕記念杯,10
7,202665060805,20260608,帯広(ばんえい),久保・姫野来場記念(2歳),10
8,202665060804,20260608,帯広(ばんえい),こもなお大爆誕記念(2歳),9
9,202665060803,20260608,帯広(ばんえい),めんどくさいたま転勤(2歳),9


In [3]:
# ── 取得件数・欠損率・重複率 サマリ ──────────────────────────
if _db.exists() and not s.get('error'):
    conn = sqlite3.connect(str(_db))
    try:
        # 月別取得件数
        _mc = conn.execute(
            "SELECT SUBSTR(race_id,1,6) AS ym, COUNT(*) AS races"
            " FROM races_ultimate GROUP BY ym ORDER BY ym DESC LIMIT 24"
        ).fetchall()
        _df_mc = pd.DataFrame(_mc, columns=['year_month','races'])
        print("=== 月別レース件数（直近24ヶ月）===")
        display(_df_mc.head(24))

        # scraped_dates の欠損確認
        _sd = conn.execute(
            "SELECT COUNT(*) AS total, SUM(no_race) AS no_race_days FROM scraped_dates"
        ).fetchone()
        print(f"\n取得済み日数: {_sd[0]}  |  非開催日: {_sd[1]}")

        # race_results_ultimate の重複チェック
        _dup = conn.execute(
            "SELECT COUNT(*) FROM (SELECT race_id, data FROM race_results_ultimate GROUP BY race_id, data HAVING COUNT(*) > 1)"
        ).fetchone()[0]
        print(f"出走結果テーブル 重複行数: {_dup}")
    finally:
        conn.close()

=== 月別レース件数（直近24ヶ月）===


,year_month,races
0,202665,192
1,202655,173
2,202654,186
3,202650,149
4,202648,168
5,202647,66
6,202646,101
7,202645,48
8,202644,117
9,202643,60



取得済み日数: 797  |  非開催日: 129


出走結果テーブル 重複行数: 0


## Simulation モード — スクレイプ実行
> `DATA_SOURCE_MODE = 'existing'` の場合はこのセルをスキップ

In [4]:
if DATA_SOURCE_MODE != 'simulation':
    print("existing モード: スクレイプはスキップします")
    print("▶ 以降のセルは simulation モード専用です。実行不要です。")


existing モード: スクレイプはスキップします
▶ 以降のセルは simulation モード専用です。実行不要です。


In [5]:
# ── simulation: スクレイプ設定 ────────────────────────────────
# フロントエンド差分: HTTP ジョブ管理なし → Python 直接呼び出し
# INV-07: jitter_sleep(≥1.0s) はそのまま動作
# INV-04: シングルスレッド asyncio ループ維持

import asyncio, threading, time, uuid, calendar
from scraping.jobs import _run_scrape_job, _scrape_jobs, _CANCEL_FLAGS, _init_jobs_db  # type: ignore
from scraping.storage import _init_sqlite_db  # type: ignore

# 期間変換 (YYYY-MM → YYYYMMDD)
def _ym_to_range(start_ym: str, end_ym: str):
    sy, sm = map(int, start_ym.split('-'))
    ey, em = map(int, end_ym.split('-'))
    start = f'{sy}{sm:02d}01'
    last  = calendar.monthrange(ey, em)[1]
    end   = f'{ey}{em:02d}{last:02d}'
    return start, end

SCRAPE_START, SCRAPE_END = _ym_to_range(SIM_SCRAPE_START, SIM_SCRAPE_END)
FORCE_RESCRAPE = False

print(f"スクレイプ期間: {SCRAPE_START} 〜 {SCRAPE_END}")
print(f"保存先 DB: {SIM_DB}")

# DB 初期化
SIM_DB_DIR.mkdir(parents=True, exist_ok=True)
_init_jobs_db()
_init_sqlite_db(SIM_DB)

17:45:02 [INFO] ================================================================================


17:45:02 [INFO] app_config ロード開始


17:45:02 [INFO] ログファイル: C:\Users\yuki2\Documents\ws\keiba-ai-pro\python-api\optuna_debug.log


17:45:02 [INFO] ================================================================================


17:45:02 [INFO] Supabase クライアント読み込み成功


17:45:02 [INFO] Supabase データ操作: 無効（認証専用モード）


スクレイプ期間: 20260101 〜 20260228
保存先 DB: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\simulation\keiba_sim.db


In [6]:
# ── simulation: ジョブ実行 ─────────────────────────────────────
# フロントエンドの useBatchScrape と同じアーキテクチャ:
#   Thread → SelectorEventLoop → _run_scrape_job()

_job_id = f'nb-sim-{uuid.uuid4().hex[:8]}'
_scrape_jobs[_job_id] = {'status': 'queued', 'progress': {}, 'result': None, 'error': None}

# ── _run_scrape_job は SIM_DB ではなく ULTIMATE_DB を内部で解決する。
# simulation では sys.path を通じて scraping.jobs 内の ULTIMATE_DB を SIM_DB に向ける方法:
# → monkeypatch で jobs モジュールの参照を書き換える
import scraping.jobs as _jobs_mod  # type: ignore
_orig_ultimate_db = getattr(_jobs_mod, '_ULTIMATE_DB_OVERRIDE', None)
# jobs.py 内は Path(__file__).parent.parent.parent / "keiba" / "data" / "keiba_ultimate.db" を参照
# simulation DB への書き込みは: storage._init_sqlite_db(SIM_DB) 済み
# ※ jobs._run_scrape_job は ULTIMATE_DB を内部で組み立てるため、
#    SIM_DB を使うには環境変数 KEIBA_DB_PATH を設定する (storage.py が参照)
import os
os.environ['KEIBA_DB_PATH'] = str(SIM_DB)  # storage.py が参照している場合

_done = threading.Event()

def _bg_sim():
    loop = asyncio.SelectorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(
            _run_scrape_job(_job_id, SCRAPE_START, SCRAPE_END, FORCE_RESCRAPE)
        )
    except Exception as e:
        _scrape_jobs[_job_id]['status'] = 'error'
        _scrape_jobs[_job_id]['error']  = str(e)
        print(f'[ERROR] {e}')
    finally:
        loop.close()
        _done.set()

print(f"ジョブ ID: {_job_id}")
print("スクレイプ開始... (Kernel 停止でキャンセル可)")
threading.Thread(target=_bg_sim, daemon=True).start()

ジョブ ID: nb-sim-e3a84882
スクレイプ開始... (Kernel 停止でキャンセル可)


17:45:02 [INFO] ━━━ スクレイプジョブ開始 ━━━ job_id=nb-sim-e3a84882 期間=20260101～20260228 force_rescrape=False


In [7]:
# ── simulation: 進捗ポーリング（5秒間隔）─────────────────────
# フロントエンド差分: useBatchScrape の 3 秒ポーリング相当
_last_msg = ''
try:
    while not _done.wait(timeout=5):
        job  = _scrape_jobs.get(_job_id, {})
        prog = job.get('progress', {})
        done_n = prog.get('done', 0)
        total_n = prog.get('total', 1)
        pct = int(done_n / total_n * 100) if total_n > 0 else 0
        msg = prog.get('message', '')
        races = prog.get('saved_races', 0)
        if msg != _last_msg:
            print(f'  [{pct:3d}%] {msg}  (レース: {races})')
            _last_msg = msg
except KeyboardInterrupt:
    _CANCEL_FLAGS[_job_id] = True
    _done.wait(timeout=30)

_job = _scrape_jobs.get(_job_id, {})
print(f"\n=== 完了: status={_job.get('status')} ===")
if _job.get('error'):
    print(f"エラー: {_job['error']}")
_res = _job.get('result') or {}
print(f"取得レース: {_res.get('races_collected', _res.get('saved_races', 0))}")

17:45:02 [INFO] カレンダー取得開始: 20260101～20260228 (59日 → 開催日のみに絞り込み)


17:45:02 [INFO] カレンダー取得: 2026/01 → 10日 ['20260104', '20260105', '20260110']


17:45:03 [INFO] カレンダー取得: 2026/02 → 9日 ['20260201', '20260207', '20260208']


17:45:04 [INFO] カレンダー絞り込み完了: 59日 → 19日（開催日のみ）


17:45:04 [INFO] 処理対象: 19日 (20260104～20260228) ※カレンダー絞り込み済み


...

17:45:08 [INFO] 20260221: 取得済み（scraped_dates）→ スキップ


17:45:08 [INFO] 20260222: 取得済み（scraped_dates）→ スキップ


17:45:08 [INFO] 20260228: 取得済み（scraped_dates）→ スキップ


17:45:08 [INFO] ━━━ スクレイプジョブ完了 ━━━ job_id=nb-sim-e3a84882 期間=20260101～20260228 保存=0レース/0頭 処理日数=19日 所要時間=0.1分



=== 完了: status=completed ===
取得レース: 0


In [8]:
# ── 収集後確認 ────────────────────────────────────────────────
s_after = db_stats(get_db_path(DATA_SOURCE_MODE))
print("=== 収集後 DB 統計 ===")
for k, v in s_after.items():
    print(f'  {k:15s}: {v}')

=== 収集後 DB 統計 ===
  races          : 52450
  results        : 575346
  latest         : 202665060812
  scraped_dates  : 797
